# 01 — Preprocessing

My part of Module 1: turning the raw OSM waterways shapefile into a clean riparian buffer and
river-line layer, for three regions — **Kasarani** (the calibration case study), **Gatharaini**,
and **Motoine**. Covers Steps 1-2 below; Steps 3-4 (Sentinel-2 composite, feature table) 


In [10]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import box

WATERWAYS_PATH = '../data/vectors/gis_osm_waterways_free_1.shp'
OUT_DIR = '../data/processed'

## Step 1 — Define the three regions

One consistent method for all three: a fixed-radius circle around a center point, not a
hand-picked bounding box. A bounding box is tempting but dangerous here — get the extent even
slightly wrong and you're screening a different population of buildings than whatever
ground-truth count you're calibrating against. One radius for every region also keeps them
comparable to each other.

- **Kasarani** — center matches Pamoja Trust's actual field-survey area.
- **Gatharaini / Motoine** — center is the geometric midpoint of the named river itself (no
  independent ground truth exists for either, so there's nothing else to match against).

In [11]:
CASE_STUDY_RADIUS_KM = 3

REGIONS = {
    'Kasarani':   {'method': 'point', 'center': (36.8969, -1.2296)},
    'Gatharaini': {'method': 'river_name', 'river_name': 'Gatharaini River'},
    'Motoine':    {'method': 'river_name', 'river_name': 'Motoine River'},
}


def get_region_center(waterways, region_key):
    cfg = REGIONS[region_key]
    if cfg['method'] == 'point':
        return cfg['center']
    river = waterways[waterways['name'] == cfg['river_name']]
    if river.empty:
        raise ValueError(f"No waterway named {cfg['river_name']!r} found in {WATERWAYS_PATH}")
    midpoint_metric = river.to_crs(epsg=32737).union_all().centroid
    midpoint = gpd.GeoSeries([midpoint_metric], crs=32737).to_crs(epsg=4326).iloc[0]
    return (midpoint.x, midpoint.y)


def get_region_aoi(center, radius_km=CASE_STUDY_RADIUS_KM):
    """A fixed-radius circle, approximated here as its bounding box just for clipping vectors."""
    lon, lat = center
    pad_deg = radius_km / 111.0
    return box(lon - pad_deg, lat - pad_deg, lon + pad_deg, lat + pad_deg)

## Step 2 — Load and clip the waterways to each region

The shapefile is Kenya-wide (47,157 features) — clip to each region's AOI before doing
anything else with it. Real flowing water only: `fclass` in `river`/`stream`, dropping drains
and canals, which aren't what a riparian buffer policy is about.

In [12]:
def load_waterways():
    return gpd.read_file(WATERWAYS_PATH)


def clip_rivers_to_aoi(waterways, aoi):
    clipped = waterways[waterways.intersects(aoi)].copy()
    rivers_only = clipped[clipped['fclass'].isin(['river', 'stream'])].copy()
    if rivers_only.empty:
        raise ValueError('No river/stream features intersect this AOI — check the AOI bounds.')
    return rivers_only

## Build the riparian buffer

Reproject to a metric CRS (EPSG:32737 — UTM 37S, correct hemisphere for Nairobi) *before*
buffering, since a buffer distance in degrees isn't a reliable metre distance. Buffer, dissolve
overlapping reaches into one shape, then reproject back to EPSG:4326 so it overlays cleanly on
standard web maps.

Also save the clipped river *lines* separately (not just the buffer polygon) in the metric
CRS — whoever calibrates a flagging distance downstream needs true distance-to-river per
building, which a single 60m buffer polygon alone can't give.

In [13]:
def build_riparian_buffer(rivers_only, buffer_m=60):
    rivers_metric = rivers_only.to_crs(epsg=32737)
    buffer_metric = rivers_metric.buffer(buffer_m)
    dissolved = gpd.GeoSeries([buffer_metric.union_all()], crs=32737)
    buffer_global = dissolved.to_crs(epsg=4326)
    return gpd.GeoDataFrame(geometry=buffer_global.explode(index_parts=False).reset_index(drop=True))


def save_river_lines(rivers_only, region_key):
    rivers_metric = rivers_only.to_crs(epsg=32737)[['name', 'fclass', 'geometry']]
    rivers_metric.to_file(f'{OUT_DIR}/{region_key.lower()}_rivers.geojson', driver='GeoJSON')

## Step 3 — Pull the Sentinel-2 composite

Full band set (`B2,B3,B4,B8,B11,B12`), not just RGB — NDVI and NDBI both need bands outside
the visible range. Cloud/shadow-masked per scene via the SCL band before compositing, so a
few contaminated pixels in an otherwise-good scene don't survive into the median.

Built from the same `center` + `CASE_STUDY_RADIUS_KM` as Step 1 — not a separately hand-picked
bounding box — so this half of the notebook can't silently disagree with the first half about
where "Kasarani" actually is.

In [14]:
import ee

ee.Initialize(project='solar-haven-349708-507818')

BANDS = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']


def _mask_s2_clouds(image):
    scl = image.select('SCL')
    mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return image.updateMask(mask)


def get_sentinel2_composite(aoi_ee, start_date, end_date, cloud_threshold=20):
    s2 = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(aoi_ee)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_threshold))
        .map(_mask_s2_clouds)
    )
    scene_count = s2.size().getInfo()
    composite = s2.select(BANDS + ['SCL']).median().clip(aoi_ee)
    return composite.select(BANDS), scene_count


def build_feature_image(composite):
    ndvi = composite.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndbi = composite.normalizedDifference(['B11', 'B8']).rename('NDBI')
    return composite.addBands(ndvi).addBands(ndbi)

## Step 4 — Build the feature table

Labelled with ESA WorldCover ground truth (real land-cover classification), not an NDVI
threshold guess — a threshold only tells you "this pixel looks vegetation-like by one number,"
not "this pixel actually is built-up."

In [15]:
def get_worldcover_builtup(aoi_ee):
    worldcover = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map').clip(aoi_ee)
    return worldcover.eq(50).rename('builtup')


def sample_feature_table(feature_image, builtup_label, aoi_ee, num_points=800, seed=42):
    training_image = feature_image.addBands(builtup_label)
    samples = training_image.stratifiedSample(
        numPoints=num_points,
        classBand='builtup',
        region=aoi_ee,
        scale=10,
        seed=seed,
        geometries=True,
    )
    features = samples.getInfo()['features']
    rows = []
    for f in features:
        row = dict(f['properties'])
        lon, lat = f['geometry']['coordinates']
        row['lon'], row['lat'] = lon, lat
        rows.append(row)
    return pd.DataFrame(rows)

## Run it for all three regions

Output file names downstream steps depend on: `{region}_riparian_buffer.geojson`,
`{region}_rivers.geojson`, and `{region}_feature_table.csv`.

In [16]:
waterways = load_waterways()
summary = {}

for region_key in REGIONS:
    print(f'--- {region_key} ---')
    center = get_region_center(waterways, region_key)
    aoi = get_region_aoi(center)
    aoi_ee = ee.Geometry.Point(list(center)).buffer(CASE_STUDY_RADIUS_KM * 1000)

    rivers_only = clip_rivers_to_aoi(waterways, aoi)
    buffer_gdf = build_riparian_buffer(rivers_only, buffer_m=60)
    buffer_gdf.to_file(f'{OUT_DIR}/{region_key.lower()}_riparian_buffer.geojson', driver='GeoJSON')
    save_river_lines(rivers_only, region_key)

    composite, scene_count = get_sentinel2_composite(aoi_ee, '2024-01-01', '2024-12-31')
    feature_image = build_feature_image(composite)
    builtup_label = get_worldcover_builtup(aoi_ee)
    feature_table = sample_feature_table(feature_image, builtup_label, aoi_ee)
    feature_table.to_csv(f'{OUT_DIR}/{region_key.lower()}_feature_table.csv', index=False)

    summary[region_key] = {
        'river_reaches_in_aoi': len(rivers_only),
        'buffer_polygons': len(buffer_gdf),
        's2_scene_count': scene_count,
        'feature_table_rows': len(feature_table),
    }
    print(summary[region_key])

pd.DataFrame(summary).T

--- Kasarani ---
{'river_reaches_in_aoi': 66, 'buffer_polygons': 4, 's2_scene_count': 22, 'feature_table_rows': 1600}
--- Gatharaini ---
{'river_reaches_in_aoi': 21, 'buffer_polygons': 2, 's2_scene_count': 22, 'feature_table_rows': 1600}
--- Motoine ---
{'river_reaches_in_aoi': 41, 'buffer_polygons': 8, 's2_scene_count': 22, 'feature_table_rows': 1600}


,river_reaches_in_aoi,buffer_polygons,s2_scene_count,feature_table_rows
Kasarani,66,4,22,1600
Gatharaini,21,2,22,1600
Motoine,41,8,22,1600
